# 팀 검수 결과 통합 및 최종 골든셋 생성

`data/evaluation/team_candidates/reviewed_*.jsonl` 파일을 모두 읽어 검증한 뒤, `READY` 문항만 `data/evaluation/golden_set_final.jsonl`로 저장합니다.

- `PENDING_REVIEW` 등 미완료 문항이 있으면 생성을 중단합니다.
- `REJECTED` 문항은 최종 골든셋에서 제외합니다.
- 중복 질문과 필수 필드 누락을 검사합니다.
- 최종 `case_id`는 `GS001`부터 다시 부여합니다.


In [1]:
import copy
import json
from collections import Counter
from pathlib import Path

# notebooks 폴더와 프로젝트 루트 어디에서 실행해도 경로가 맞도록 설정
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVALUATION = ROOT / "data" / "evaluation"
TEAM_DIR = EVALUATION / "team_candidates"
FINAL_PATH = EVALUATION / "golden_set_final.jsonl"

REVIEWED_FILE_PATTERN = "reviewed_*.jsonl"
EXCLUDED_REVIEWED_FILES = {"reviewed_통합.jsonl"}
OVERWRITE_FINAL = False

print("검수 파일 폴더:", TEAM_DIR)
print("최종 파일:", FINAL_PATH)


검수 파일 폴더: c:\Users\Playdata\Desktop\mle-01-p3-team2\data\evaluation\team_candidates
최종 파일: c:\Users\Playdata\Desktop\mle-01-p3-team2\data\evaluation\golden_set_final.jsonl


In [2]:
def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def write_jsonl(path, rows, overwrite=False):
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"{path}이 이미 있습니다. 기존 파일을 확인하거나 "
            "OVERWRITE_FINAL=True로 바꾸세요."
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    temporary_path.replace(path)


def ensure_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def add_terminal_punctuation(text):
    text = str(text).strip()
    if text and text[-1] not in ".!?。！？":
        text += "."
    return text


def make_reference(row):
    sentences = [
        add_terminal_punctuation(fact)
        for fact in ensure_list(row.get("expected_facts"))
        if str(fact).strip()
    ]
    if row.get("expected_behavior") == "ANSWER_WITH_WARNING":
        sentences.extend(
            add_terminal_punctuation(assertion)
            for assertion in ensure_list(row.get("safety_assertions"))
            if str(assertion).strip()
        )
    return " ".join(dict.fromkeys(sentences))


In [3]:
reviewed_files = [
    path
    for path in sorted(TEAM_DIR.glob(REVIEWED_FILE_PATTERN))
    if path.name not in EXCLUDED_REVIEWED_FILES
]
if not reviewed_files:
    raise FileNotFoundError(
        f"{TEAM_DIR}에 {REVIEWED_FILE_PATTERN} 파일이 없습니다."
    )

reviewed_candidates = []
file_counts = {}
unfinished = []

for path in reviewed_files:
    rows = read_jsonl(path)
    file_counts[path.name] = len(rows)
    for row in rows:
        status = row.get("status")
        if status not in {"READY", "REJECTED"}:
            unfinished.append(
                f"{path.name}:{row.get('case_id', 'case_id 없음')}"
            )
        source = copy.deepcopy(row)
        source["source_reviewed_file"] = path.name
        reviewed_candidates.append(source)

if unfinished:
    raise ValueError(
        "아직 검수하지 않은 문항이 있습니다: "
        + ", ".join(unfinished)
    )

print("불러온 검수 파일")
for path in reviewed_files:
    print("-", path.name, file_counts[path.name])
print("전체 검수 문항:", len(reviewed_candidates))


ValueError: 아직 검수하지 않은 문항이 있습니다: reviewed_수민.jsonl:수민__CAND_0006, reviewed_수민.jsonl:수민__CAND_0010

In [ ]:
approved_candidates = [
    row
    for row in reviewed_candidates
    if row.get("status") == "READY"
]
rejected_candidates = [
    row
    for row in reviewed_candidates
    if row.get("status") == "REJECTED"
]

FINAL_KEYS = [
    "case_id",
    "question",
    "reference",
    "question_type",
    "difficulty",
    "query_strategy",
    "expected_behavior",
    "expected_facts",
    "expected_entity_ids",
    "expected_relation_ids",
    "expected_evidence_ids",
    "expected_source_chunk_ids",
    "safety_assertions",
    "status",
]
REQUIRED_KEYS = {
    "case_id",
    "question",
    "reference",
    "query_strategy",
    "expected_behavior",
    "expected_facts",
    "expected_entity_ids",
    "expected_relation_ids",
    "expected_evidence_ids",
    "expected_source_chunk_ids",
    "safety_assertions",
    "status",
}

final_golden = []
for number, candidate in enumerate(approved_candidates, start=1):
    source = copy.deepcopy(candidate)
    source["reference"] = make_reference(source)
    row = {
        key: source[key]
        for key in FINAL_KEYS
        if key in source
    }
    row["case_id"] = f"GS{number:03d}"
    row["status"] = "READY"
    final_golden.append(row)

question_counts = Counter(
    row["question"].strip()
    for row in final_golden
)
duplicate_questions = sorted(
    question
    for question, count in question_counts.items()
    if count > 1
)
if duplicate_questions:
    raise ValueError(
        f"중복 질문이 있습니다: {duplicate_questions}"
    )

for row in final_golden:
    missing_keys = REQUIRED_KEYS - set(row)
    if missing_keys:
        raise ValueError(
            f"{row['case_id']} 필수 키 누락: {sorted(missing_keys)}"
        )
    if not row["question"].strip():
        raise ValueError(f"{row['case_id']} 질문이 비어 있습니다.")
    if not row["expected_facts"]:
        raise ValueError(f"{row['case_id']} 기대 사실이 비어 있습니다.")
    if not row["reference"].strip():
        raise ValueError(f"{row['case_id']} reference가 비어 있습니다.")
    if (
        row["expected_behavior"] in {"ANSWER", "ANSWER_WITH_WARNING"}
        and not row["expected_evidence_ids"]
    ):
        raise ValueError(f"{row['case_id']} 근거 ID가 비어 있습니다.")

write_jsonl(
    FINAL_PATH,
    final_golden,
    overwrite=OVERWRITE_FINAL,
)

print("전체 검수 문항:", len(reviewed_candidates))
print("검수 통과 문항:", len(final_golden))
print("제외 문항:", len(rejected_candidates))
print("최종 파일:", FINAL_PATH)


전체 검수 문항: 61
검수 통과 문항: 58
제외 문항: 1
최종 파일: c:\Project_AI\mle-01-p3-team2\data\evaluation\golden_set_final.jsonl
